# R4: train the MobileNetV2 eye-state CNN on Kaggle

**Before running:** in the right-hand panel set **Accelerator = GPU T4 x2 (or P100)**, turn **Internet = On**
(needs a phone-verified Kaggle account), and **Add Input** -> your private dataset `mrl-eye`.

All code lives in the GitHub repo; this notebook only clones it and runs the scripts, so every change
to the model is a real commit. Set `BRANCH` to the branch you pushed.

In [ ]:
BRANCH = "r4/interface-contract"   # <- the branch that contains your R4 commits
REPO = "https://github.com/aarika001/Real-Time-Driver-Fatigue-and-Distraction-Alert-System.git"
!nvidia-smi -L
%cd /kaggle/working
!rm -rf repo && git clone -q --depth 1 -b $BRANCH $REPO repo
%cd repo
!git log --oneline -5

In [ ]:
# locate the MRL Eye folder (Kaggle unzips uploads automatically) or the zip
import glob, os
hits = glob.glob('/kaggle/input/**/mrlEyes_2018_01', recursive=True) or glob.glob('/kaggle/input/**/*.zip', recursive=True)
MRL = hits[0] if hits else None
print('MRL root:', MRL)
assert MRL, 'Add your mrl-eye dataset with Add Input in the right panel'
os.environ['MRL'] = MRL

**Step 1 - tests (about 1 minute).** Includes a tiny training run on synthetic images. Everything must pass before the real run.

In [ ]:
!pip -q install pyyaml pytest onnx onnxruntime
!python -m pytest tests/r4 -q

**Step 2 - full training (about 15-25 minutes on a T4).** Prints validation metrics every epoch, then the test report on unseen people.

In [ ]:
!python scripts/r4/train_eye_cnn.py --mrl_root "$MRL" --out_dir /kaggle/working/eye_cnn

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv('/kaggle/working/eye_cnn/history.csv')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.epoch, h.train_loss, marker='o'); ax[0].set_title('training loss'); ax[0].set_xlabel('epoch')
ax[1].plot(h.epoch, h.val_auroc, marker='o', label='val AUROC'); ax[1].plot(h.epoch, h.val_subject_f1, marker='o', label='val subject-F1')
ax[1].set_xlabel('epoch'); ax[1].legend(); plt.tight_layout(); plt.savefig('/kaggle/working/eye_cnn/training_curves.png', dpi=120); plt.show()
print(open('/kaggle/working/eye_cnn/test_report.md').read())

**Step 3 - ONNX export + evaluation on the GPU (about 5 minutes).** Per-condition results and stress sweeps on the held-out people.

In [ ]:
!python scripts/r4/export_onnx.py --checkpoint /kaggle/working/eye_cnn/best.pt --out /kaggle/working/eye_cnn/eye_cnn.onnx
!python scripts/r4/eval_eye_cnn.py --eye_model /kaggle/working/eye_cnn/best.pt --splits /kaggle/working/eye_cnn/splits.csv --mrl_root "$MRL" --stress_images 5000 --out_dir /kaggle/working/eye_cnn/eye_eval

**Step 4 - download.** Output panel -> `eye_cnn_results.zip` (contains best.pt, eye_cnn.onnx, reports, plots). Keep the model files OUT of git (share via Drive); reports and plots can go into results/r4/.

In [ ]:
!cd /kaggle/working && zip -q -r eye_cnn_results.zip eye_cnn -x 'eye_cnn/last.pt' && ls -lh eye_cnn_results.zip